# Course 2 — Bias / Variance & Learning Curves
==============================================

Diagnosing whether a model suffers from high bias (underfitting) or high variance (overfitting) is crucial for deciding how to improve performance. This notebook uses learning curves and validation curves to perform this diagnosis.

### In this notebook, we will explore:
1. **Learning Curves**: Plotting model accuracy against the size of the training dataset to identify bias and variance limits.
2. **Validation Curves**: Visualizing how a model's performance scales with a model hyperparameter (such as the RBF kernel coefficient $\gamma$).
3. **Bias/Variance Diagnosis**: Formulating systematic remedies for model improvements.

In [ ]:
import sys
from pathlib import Path
# Add repository root to sys.path dynamically
project_root = Path(".").resolve()
while project_root.name and not (project_root / "utils").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons


## 1. Data Preparation

We split and scale synthetic moon-shaped data. Scaling is vital for SVMs to prevent attributes with larger scales from dominating the distance metric.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 2. Learning Curves

A **Learning Curve** plots the training and validation scores as a function of the number of training examples ($m$).

- **High Bias (Underfitting)**: Adding more data does not help. The training and validation errors plateau early and lie close to each other at a high error rate (low accuracy).
- **High Variance (Overfitting)**: A large gap remains between the training and validation curves. The training accuracy remains high, but validation accuracy lags behind. Adding more training data usually helps the validation accuracy catch up.

In [ ]:
# ── Learning Curve (training set size vs error) ───────────────────────

print("── Learning Curves ──")

train_sizes, train_scores, val_scores = learning_curve(
    SVC(kernel="rbf", gamma=0.1),
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring="accuracy",
    random_state=42,
)

train_mean = np.mean(train_scores, axis=1)
val_mean   = np.mean(val_scores, axis=1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_sizes, train_mean, "o-", label="Train accuracy")
plt.plot(train_sizes, val_mean, "o-", label="Validation accuracy")
plt.xlabel("Training examples")
plt.ylabel("Accuracy")
plt.title("Learning Curve — RBF Kernel SVM")
plt.legend()
plt.grid(alpha=0.3)

## 3. Validation Curves

A **Validation Curve** plots the training and validation scores against a model hyperparameter (in this case, $\gamma$ in an RBF-kernel SVM).

The hyperparameter $\gamma$ controls how far the influence of a single training example reaches:
- **Small $\gamma$ (High Bias)**: The model is highly constrained, yielding broad, simple decision boundaries.
- **Large $\gamma$ (High Variance)**: The model is overly flexible, fitting complex boundary contours around individual points (overfitting).

In [ ]:
# ── Validation Curve (hyperparameter vs error) ────────────────────────

print("── Validation Curve (gamma in RBF SVM) ──")

param_range = np.logspace(-3, 2, 20)
train_scores, val_scores = validation_curve(
    SVC(kernel="rbf"),
    X_train, y_train,
    param_name="gamma",
    param_range=param_range,
    cv=5,
    scoring="accuracy",
    )

train_mean = np.mean(train_scores, axis=1)
val_mean   = np.mean(val_scores, axis=1)

plt.subplot(1, 2, 2)
plt.semilogx(param_range, train_mean, "o-", label="Train accuracy")
plt.semilogx(param_range, val_mean, "o-", label="Validation accuracy")
plt.xlabel("gamma (RBF kernel)")
plt.ylabel("Accuracy")
plt.title("Validation Curve — Bias/Variance Diagnosis")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Model Diagnosis

We identify the optimal hyperparameter $\gamma$ that maximizes our 5-fold cross-validation accuracy.

In [ ]:
# ── Diagnosis ─────────────────────────────────────────────────────────

best_gamma = param_range[np.argmax(val_mean)]
print(f"Best gamma: {best_gamma:.4f}")

## Key Takeaways

- **High Bias (Underfitting)**: Both training and validation accuracy are low.
  - *Remedies*: Choose a larger, more expressive model (e.g. increase polynomial degree, add hidden units), add more features, or decrease regularization strength ($\lambda \downarrow$).
- **High Variance (Overfitting)**: Training accuracy is high, but validation accuracy is significantly lower.
  - *Remedies*: Add more training examples, simplify model complexity, select a smaller subset of features, or increase regularization ($\lambda \uparrow$).
- **Learning Curves**:
  - High Bias: Curves converge early at low performance. Adding data does **not** help.
  - High Variance: Significant gap between curves. Adding data **helps** close the gap.
- **Validation Curves**: Useful to sweep parameters (like regularizer weight or SVM kernel widths) to select the optimal model size.

In [ ]:
print("""
╔══ Key Takeaways ───────────────────────────────────────────────╗
║ • High Bias (Underfitting): both train & val accuracy are low  ║
║   → Solutions: bigger model, add features, decrease L2/L1 reg  ║
║ • High Variance (Overfitting): high train & low val accuracy   ║
║   → Solutions: more data, add regularization, simplify model   ║
║ • Learning Curves: plot training size vs. performance          ║
║   → High Bias: train & val plateau early close to each other   ║
║   → High Variance: gap remains between train & val curves      ║
║ • Validation Curves: sweep hyperparameter (e.g. λ, γ, degree)  ║
║   → Pick parameter value that maximizes validation accuracy    ║
╚════════════════════════════════════════════════════════════════╝
""")